# Triton Fused Kernels — Benchmark

Reproduces the benchmarks from the [triton-fused-kernels](https://github.com/shiva-sankeerth/triton-fused-kernels) repo on a T4 GPU.

**Before running**: Runtime → Change runtime type → **T4 GPU**

| Cell | What it does |
|---|---|
| 1 | Clone repo and install dependencies |
| 2 | Correctness validation — Triton vs PyTorch numerics |
| 3 | Full benchmark sweep → results CSV |

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────
# Verify GPU
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No GPU found — change runtime type to T4"
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

# Clone the repo
import os
REPO_URL = "https://github.com/shiva-sankeerth/triton-fused-kernels.git"
REPO_DIR = "triton-fused-kernels"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet

import triton
print(f"Triton: {triton.__version__}")
print("\nSetup complete.")

In [ ]:
# ── Cell 2: Correctness Validation ───────────────────────────────────
# Verify Triton kernels match PyTorch baselines numerically.
# All assertions must pass before running the benchmark.

import sys
sys.path.insert(0, ".")

import torch
from kernels.rms_norm import rms_norm_triton
from kernels.swiglu import swiglu_triton
from kernels.rms_norm_quant import rms_norm_quant_triton
from baselines.rms_norm import rms_norm_pytorch
from baselines.swiglu import swiglu_pytorch

device = "cuda"
torch.manual_seed(42)
PASS_THRESHOLD = 1e-4
all_pass = True

print("=" * 55)
print("RMSNorm")
print("=" * 55)
for hidden_size in [512, 1024, 2048, 4096, 8192]:
    x = torch.randn(128, hidden_size, device=device)
    w = torch.ones(hidden_size, device=device)
    y_triton  = rms_norm_triton(x.contiguous(), w.contiguous())
    y_pytorch = rms_norm_pytorch(x, w)
    max_diff  = (y_triton - y_pytorch).abs().max().item()
    ok = max_diff < PASS_THRESHOLD
    all_pass &= ok
    print(f"  hidden={hidden_size:5d}  max_diff={max_diff:.2e}  {'PASS' if ok else 'FAIL !!'}")

print()
print("=" * 55)
print("SwiGLU")
print("=" * 55)
for hidden_size in [512, 1024, 2048, 4096, 8192]:
    gate = torch.randn(128, hidden_size, device=device)
    up   = torch.randn(128, hidden_size, device=device)
    y_triton  = swiglu_triton(gate.contiguous(), up.contiguous())
    y_pytorch = swiglu_pytorch(gate, up)
    max_diff  = (y_triton - y_pytorch).abs().max().item()
    ok = max_diff < PASS_THRESHOLD
    all_pass &= ok
    print(f"  hidden={hidden_size:5d}  max_diff={max_diff:.2e}  {'PASS' if ok else 'FAIL !!'}")

print()
print("=" * 55)
print("RMSNorm + INT8 Quantization")
print("=" * 55)
for hidden_size in [512, 1024, 2048, 4096, 8192]:
    x = torch.randn(128, hidden_size, device=device)
    w = torch.ones(hidden_size, device=device)
    q_triton, scale_triton = rms_norm_quant_triton(x.contiguous(), w.contiguous())
    # Reference: apply rms_norm then quantize
    normed = rms_norm_pytorch(x, w)
    max_abs = normed.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8)
    scale_ref = (max_abs / 127.0).squeeze(-1)
    q_ref = (normed / max_abs * 127).clamp(-128, 127).round().to(torch.int8)
    max_diff = (q_triton.float() - q_ref.float()).abs().max().item()
    scale_err = (scale_triton - scale_ref).abs().max().item()
    ok = max_diff <= 1 and scale_err < 1e-5
    all_pass &= ok
    print(f"  hidden={hidden_size:5d}  q_diff={max_diff:.0f} step(s)  scale_err={scale_err:.2e}  {'PASS' if ok else 'FAIL !!'}")

print()
if all_pass:
    print("All validations PASSED. Ready to benchmark.")
else:
    print("SOME VALIDATIONS FAILED — do not run benchmark until fixed.")
    raise RuntimeError("Validation failed")

In [ ]:
# ── Cell 3: Run Benchmark ────────────────────────────────────────────
# Triton will JIT-compile kernels on first call (~30-60s).
# Full benchmark takes ~2-5 minutes on T4.

!python benchmark/run.py --dtype float32

# Optional: also run float16 variant
# !python benchmark/run.py --dtype float16

# Show what was saved
!ls -lh benchmark/results/